# AWAS Real-Time Violation Dashboard

**Author:** Haruto Iriyama

**Purpose.** This notebook queries the `violations` collection in MongoDB and renders interactive dashboards that **update in real time** as the Spark streaming pipeline writes new records.

## Charts produced (all live, interactive)

| # | Visualisation | Chart in this notebook |
|---|---|---|
| 1 | Violation count over time | Chart 1 (dual-axis time series — bar trace, left axis) |
| 2 | Average violation speed over time | Chart 1 (dual-axis time series — line trace, right axis) + 5-bucket moving average |
| 3 | Separate lines for instantaneous and average-speed violations | Chart 2 (stacked area) |
| 4 | Camera-level violation counts | Chart 3 (per-camera bar chart, by **ending** camera) |
| 5 | Segment-level average-speed violation counts | Chart 4 (per-segment bar chart, A→B and B→C) |
| 6 | Annotated peaks, sudden drops, unusual periods | Annotations + dynamic percentile lines + shaded spike/drop regions on Chart 1 |
| 7 | Geographic clustering | Chart 5 (Folium map with camera markers, segment polylines coloured by violation density) |

## Key design choices

* **Interactive Plotly** charts with hover, zoom, and pan support.
* **Dynamic percentile reference lines (p50 / p75 / p90 / p95)** computed from the current data — adapts automatically to traffic regime without hardcoded thresholds.
* **Rolling z-score** for spike/drop detection — also data-adaptive.
* **Segment-level polylines on the Folium map** — links camera-level (Chart 3) and segment-level (Chart 4) data visually on a real OpenStreetMap basemap.

## Implementation note
A `time.sleep` loop with `clear_output(wait=True)` gives a smooth, controllable refresh that does not require a Mongo change-stream (change streams require a replica set which may not be available in all environments). The Plotly renderer is set to `'notebook_connected'` so figures survive the periodic `clear_output` — this addresses a known rendering issue in Jupyter where Plotly mounts disappear after clear.

In [7]:
# --- Imports -----------------------------------------------------------------
import datetime as dt
import json
import os
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import folium
from IPython.display import display, clear_output, HTML
from pymongo import MongoClient

# 'notebook_connected' loads plotly.js from a CDN once per kernel session
# and re-mounts cleanly on every clear_output refresh cycle.
pio.renderers.default = 'notebook_connected'

print('Plotly renderer:', pio.renderers.default)

Plotly renderer: notebook_connected


In [8]:
# --- Parameters --------------------------------------------------------------
# HOST_IP = LAN IPv4 of the host running MongoDB.
HOST_IP = '192.168.100.81'

MONGO_URI         = f'mongodb://{HOST_IP}:27017'
MONGO_DB          = 'awas'
MONGO_VIOLATIONS  = 'violations'
MONGO_CAMERAS     = 'cameras'

REFRESH_SECONDS   = 5            # how often to redraw the charts
BUCKET            = '5s'         # initial bucket; auto-widens if data volume is high
LOOKBACK_DAYS     = None         # None = all data; e.g. 7 = last week of *event-date* days
MAX_REFRESHES     = 360          # safety cap (~30 min @ 5 s); set higher for marathon demos

client = MongoClient(f"mongodb://{HOST_IP}:27017/")
db = client[MONGO_DB]
violations_coll = db[MONGO_VIOLATIONS]
cameras_coll    = db[MONGO_CAMERAS]
client.admin.command('ping')
print(f'AWAS_HOST       = {HOST_IP!r}')
print(f'MONGO_URI       = {MONGO_URI}')
print(f'REFRESH_SECONDS = {REFRESH_SECONDS}')
print(f'BUCKET          = {BUCKET} (auto-widens if >1000 buckets)')
print(f'MAX_REFRESHES   = {MAX_REFRESHES}  (~{MAX_REFRESHES * REFRESH_SECONDS // 60} min cap)')

AWAS_HOST       = '192.168.100.81'
MONGO_URI       = mongodb://192.168.100.81:27017
REFRESH_SECONDS = 5
BUCKET          = 5s (auto-widens if >1000 buckets)
MAX_REFRESHES   = 360  (~30 min cap)


## §0. Sanity check — is there data to visualise?

Before we start the live loop, this cell verifies that the streaming pipeline has actually written something to MongoDB. The dashboard renders "No violations yet" if Mongo is empty, but if you see that AND you expect data, this cell tells you exactly what's wrong.

In [9]:
# --- Sanity check: does Mongo have data? -------------------------------------
n_daily = violations_coll.count_documents({})
n_indiv = next(
    violations_coll.aggregate([{'$unwind': '$violations'}, {'$count': 'n'}]),
    {'n': 0}
)['n']
n_cam   = cameras_coll.count_documents({})

print(f'cameras collection:        {n_cam} docs')
print(f'violations daily docs:     {n_daily}')
print(f'individual violations:     {n_indiv}')

if n_indiv == 0:
    print()
    print('=========================================================================')
    print('  NO VIOLATIONS YET. The dashboard will render an empty header.')
    print('  Things to check, in order:')
    print('  1. Is data_design_streaming.ipynb running? (§11 query.isActive == True)')
    print('  2. Are producer A/B/C notebooks still logging "published batch_id=..."?')
    print('  3. Does AWAS_HOST match between this notebook and the streaming notebook?')
    print('  4. Spark UI at http://localhost:4040 - is the awas-violations-sink query green?')
    print('=========================================================================')
else:
    sample = next(violations_coll.aggregate([
        {'$unwind': '$violations'}, {'$limit': 1},
    ]))
    print()
    print('Sample violation (first one in Mongo):')
    print(json.dumps(sample, default=str, indent=2)[:600])

cameras collection:        3 docs
violations daily docs:     2142
individual violations:     2777

Sample violation (first one in Mongo):
{
  "_id": "6a146a91332fac800446d67a",
  "car_plate": "CJW 924",
  "date": "2024-01-01 00:00:00",
  "created_at": "2026-05-25 15:28:17.780000",
  "expires_at": "2028-05-24 15:35:09.632000",
  "updated_at": "2026-05-25 15:35:09.632000",
  "violations": {
    "violation_id": "48d36fc2e375055ba14d9de6b5bafaff065dbef8dd34c62b31fadb4c18031208",
    "violation_type": "INSTANTANEOUS",
    "camera_id_start": 1,
    "camera_id_end": 1,
    "timestamp_start": "2024-01-01 08:00:01",
    "timestamp_end": "2024-01-01 08:00:01",
    "speed_reading": 148.3,
    "speed_limit": 110.0,
    "event_id_start": "d0


## §1. Helpers — data access, aggregation, map building

In [10]:
def fetch_violations(coll, since=None):
    """Flatten every embedded violation in the daily docs into one tidy DataFrame.

    Parameters
    ----------
    coll : pymongo.collection.Collection
        The violations collection.
    since : datetime, optional
        If given, only daily docs whose ``date`` is >= since are read.

    Returns
    -------
    pandas.DataFrame
        One row per individual violation, with timestamps already parsed.
    """
    match = {} if since is None else {'date': {'$gte': since}}
    pipeline = [
        {'$match': match},
        {'$unwind': '$violations'},
        {'$project': {
            '_id': 0,
            'car_plate': 1,
            'date': 1,
            'violation_id':      '$violations.violation_id',
            'violation_type':    '$violations.violation_type',
            'camera_id_start':   '$violations.camera_id_start',
            'camera_id_end':     '$violations.camera_id_end',
            'timestamp_start':   '$violations.timestamp_start',
            'timestamp_end':     '$violations.timestamp_end',
            'speed_reading':     '$violations.speed_reading',
            'speed_limit':       '$violations.speed_limit',
            'producer_id_start': '$violations.producer_id_start',
            'producer_id_end':   '$violations.producer_id_end',
        }},
        {'$sort': {'timestamp_start': 1}},
    ]
    rows = list(coll.aggregate(pipeline))
    if not rows:
        return pd.DataFrame(columns=[
            'car_plate', 'date', 'violation_id', 'violation_type',
            'camera_id_start', 'camera_id_end',
            'timestamp_start', 'timestamp_end',
            'speed_reading', 'speed_limit',
            'producer_id_start', 'producer_id_end',
        ])
    df = pd.DataFrame(rows)
    df['timestamp_start'] = pd.to_datetime(df['timestamp_start'])
    df['timestamp_end']   = pd.to_datetime(df['timestamp_end'])
    return df


def fetch_camera_meta(coll):
    """Load the static cameras collection into a tidy DataFrame for the Folium map.

    Parameters
    ----------
    coll : pymongo.collection.Collection
        The cameras collection (populated by §3.5 of the streaming notebook).

    Returns
    -------
    pandas.DataFrame
        One row per camera, sorted by ``camera_id`` ascending, with columns
        ``camera_id``, ``lat``, ``lon``, ``speed_limit``, ``position``. The
        GeoJSON ``location`` field is flattened into separate ``lat`` and
        ``lon`` columns for direct use by Folium markers. Returns an empty
        DataFrame with the same columns if the collection is empty (the
        downstream map renderer handles that case gracefully).
    """
    rows = list(coll.find({}, {'_id': 0}).sort('camera_id', 1))
    if not rows:
        return pd.DataFrame(columns=['camera_id', 'lat', 'lon', 'speed_limit', 'position'])
    df = pd.DataFrame(rows)
    df['lon'] = df['location'].apply(lambda g: g['coordinates'][0])
    df['lat'] = df['location'].apply(lambda g: g['coordinates'][1])
    return df[['camera_id', 'lat', 'lon', 'speed_limit', 'position']]


def choose_bucket(df, target_buckets=240, default_bucket='5s'):
    """Adaptive bucket size: widen when there are too many buckets.

    Plotly slows down past ~1000 markers per trace. We aim for ~240 buckets
    (~20 min @ 5 s, ~4 h @ 1 min), choosing the next-larger step when the data
    span demands it.
    """
    if df.empty:
        return default_bucket
    span = df['timestamp_start'].max() - df['timestamp_start'].min()
    seconds = max(span.total_seconds(), 1)
    target_seconds = seconds / target_buckets
    # Step ladder, smallest first.
    for label, secs in [('5s', 5), ('15s', 15), ('30s', 30),
                        ('1min', 60), ('5min', 300), ('15min', 900),
                        ('1H', 3600), ('6H', 21600), ('1D', 86400)]:
        if secs >= target_seconds:
            return label
    return '1D'


def aggregate_over_time(df, bucket):
    """Resample violations into time buckets for the line chart.

    Returns a DataFrame with: bucket, count_total, count_inst, count_avg,
    avg_speed, max_speed, p50, p75, p90, p95, and a 5-bucket moving average.
    """
    cols = ['bucket', 'count_total', 'count_inst', 'count_avg',
            'avg_speed', 'max_speed', 'p50', 'p75', 'p90', 'p95', 'avg_speed_ma5']
    if df.empty:
        return pd.DataFrame(columns=cols)
    by_bucket = df.set_index('timestamp_start').groupby(pd.Grouper(freq=bucket))
    out = pd.DataFrame({
        'count_total': by_bucket['violation_id'].count(),
        'count_inst':  by_bucket.apply(lambda g: (g['violation_type'] == 'INSTANTANEOUS').sum()),
        'count_avg':   by_bucket.apply(lambda g: (g['violation_type'] == 'AVERAGE').sum()),
        'avg_speed':   by_bucket['speed_reading'].mean(),
        'max_speed':   by_bucket['speed_reading'].max(),
        'p50':         by_bucket['speed_reading'].quantile(0.50),
        'p75':         by_bucket['speed_reading'].quantile(0.75),
        'p90':         by_bucket['speed_reading'].quantile(0.90),
        'p95':         by_bucket['speed_reading'].quantile(0.95),
    }).reset_index().rename(columns={'timestamp_start': 'bucket'})
    out = out[out['count_total'] > 0].reset_index(drop=True)
    # Moving average of avg_speed (sample plot style — Image 2 in spec).
    out['avg_speed_ma5'] = out['avg_speed'].rolling(window=5, min_periods=1).mean()
    return out


def find_interesting_points(agg):
    """Identify extrema and z-score-based anomalies in the aggregated series.

    Returns a dict with: max_count, min_count, max_speed, min_speed,
    spikes (>= +2 sigma rolling), drops (<= -2 sigma rolling).
    """
    if agg.empty:
        return {}
    a = agg.copy()
    rolling = a['count_total'].rolling(window=5, min_periods=2)
    a['count_mean'] = rolling.mean()
    a['count_std']  = rolling.std().fillna(0)
    a['z'] = np.where(a['count_std'] > 0,
                      (a['count_total'] - a['count_mean']) / a['count_std'], 0.0)
    return {
        'max_count': a.loc[[a['count_total'].idxmax()]],
        'min_count': a.loc[[a['count_total'].idxmin()]],
        'max_speed': a.loc[[a['avg_speed'].idxmax()]] if a['avg_speed'].notna().any() else pd.DataFrame(),
        'min_speed': a.loc[[a['avg_speed'].idxmin()]] if a['avg_speed'].notna().any() else pd.DataFrame(),
        'spikes':    a[a['z'] >= 2.0],
        'drops':     a[a['z'] <= -2.0],
    }


def build_camera_map(camera_df, viol_df):
    """Render a Folium map: camera pin markers + segment polylines coloured by density.

    Uses CartoDB Positron tiles for a clean, uncluttered basemap. Camera
    locations are marked with custom teardrop-pin DivIcons (colour-coded by
    violation count) that match the university sample layout. Segment polylines
    connect consecutive cameras and are coloured by AVERAGE-speed violation
    density. A professional legend is rendered in the top-right corner.

    Parameters
    ----------
    camera_df : pandas.DataFrame
        Camera metadata — columns: camera_id, lat, lon, speed_limit, position.
    viol_df : pandas.DataFrame
        Flattened violation records — columns include violation_id,
        violation_type, camera_id_end, camera_id_start.

    Returns
    -------
    folium.Map
        Fully rendered interactive map ready for ``_repr_html_()``.
    """
    # --- Graceful empty-data fallback ----------------------------------------
    if camera_df.empty:
        return folium.Map(
            location=[2.16, 102.65], zoom_start=14, tiles='CartoDB positron'
        )

    centre_lat = camera_df['lat'].mean()
    centre_lon = camera_df['lon'].mean()

    # CartoDB Positron: light grey/white background that lets coloured data
    # stand out clearly — matches the university sample tile style.
    fmap = folium.Map(
        location=[centre_lat, centre_lon],
        zoom_start=15,
        tiles='CartoDB positron',
    )

    # --- Violation-count aggregation -----------------------------------------
    cam_counts = (
        viol_df.groupby('camera_id_end')['violation_id'].count()
        if not viol_df.empty else pd.Series(dtype=int)
    )
    avg_only = (
        viol_df[viol_df['violation_type'] == 'AVERAGE']
        if not viol_df.empty else pd.DataFrame()
    )
    seg_counts = (
        avg_only.groupby(['camera_id_start', 'camera_id_end'])['violation_id'].count()
        if not avg_only.empty else pd.Series(dtype=int)
    )

    # --- Colour ramps (hex codes used throughout for consistency) ------------
    def marker_hex(c):
        """Return hex colour for a camera marker based on violation count."""
        if c <= 200:   return '#2ca02c'   # green
        if c <= 1000:  return '#ff7f0e'   # orange
        return '#d62728'                   # red

    def segment_hex(c):
        """Return hex colour for a segment polyline based on AVG violation count."""
        if c <= 50:   return '#2ca02c'
        if c <= 200:  return '#ff7f0e'
        return '#d62728'

    # --- Camera pin markers (teardrop DivIcon with camera emoji) -------------
    sorted_cams = camera_df.sort_values('position').reset_index(drop=True)

    for r in sorted_cams.itertuples():
        c     = int(cam_counts.get(int(r.camera_id), 0))
        color = marker_hex(c)

        # Teardrop pin: a rotated rounded square forms the classic map-pin
        # shape. The camera emoji is layered on top, unrotated, centred in
        # the upper circle of the teardrop. icon_anchor places the bottom
        # tip of the pin exactly on the camera's lat/lon coordinate.
        pin_html = f"""
        <div style="width:34px; height:44px; position:relative;">
          <div style="
            width: 30px; height: 30px;
            background: {color};
            border-radius: 50% 50% 50% 0;
            transform: rotate(-45deg);
            position: absolute; top: 0; left: 2px;
            box-shadow: 2px 3px 8px rgba(0,0,0,0.30);
          "></div>
          <div style="
            position: absolute; top: 3px; left: 4px;
            width: 26px; height: 26px;
            display: flex; align-items: center; justify-content: center;
            font-size: 14px; line-height: 1; z-index: 1;
          ">&#128247;</div>
        </div>
        """

        # Popup: clean HTML table matching the university sample layout.
        popup_html = f"""
        <div style="
          font-family: Arial, sans-serif; font-size: 13px;
          min-width: 190px; line-height: 1.6;
        ">
          <div style="
            font-size: 15px; font-weight: bold;
            color: {color}; margin-bottom: 6px;
            border-bottom: 2px solid {color}; padding-bottom: 4px;
          ">Camera {int(r.camera_id)}</div>
          <table style="width:100%; border-collapse:collapse;">
            <tr>
              <td style="color:#666; padding:2px 4px 2px 0;">Speed Limit</td>
              <td style="text-align:right; font-weight:bold; padding:2px 0;">
                {r.speed_limit:g} km/h
              </td>
            </tr>
            <tr>
              <td style="color:#666; padding:2px 4px 2px 0;">Violations</td>
              <td style="text-align:right; font-weight:bold; padding:2px 0; color:{color};">
                {c}
              </td>
            </tr>
            <tr>
              <td style="color:#666; padding:2px 4px 2px 0;">Date</td>
              <td style="text-align:right; padding:2px 0;">All dates</td>
            </tr>
          </table>
        </div>
        """

        folium.Marker(
            location=[r.lat, r.lon],
            icon=folium.DivIcon(
                html=pin_html,
                icon_size=(34, 44),
                icon_anchor=(17, 44),   # anchor at the bottom tip of the pin
            ),
            popup=folium.Popup(popup_html, max_width=220),
            tooltip=f'Camera {int(r.camera_id)} — {c} violations',
        ).add_to(fmap)

    # --- Segment polylines coloured by AVERAGE-violation density -------------
    for i in range(len(sorted_cams) - 1):
        a_cam = sorted_cams.iloc[i]
        b_cam = sorted_cams.iloc[i + 1]
        key = (int(a_cam.camera_id), int(b_cam.camera_id))
        c = int(seg_counts.get(key, 0)) if not seg_counts.empty else 0
        folium.PolyLine(
            locations=[[a_cam.lat, a_cam.lon], [b_cam.lat, b_cam.lon]],
            color=segment_hex(c),
            weight=5,
            opacity=0.85,
            tooltip=(
                f'Segment {key[0]}→{key[1]}: {c} average-speed violations'
            ),
        ).add_to(fmap)

    # --- Professional legend (top-right, matching university sample) ---------
    # Positioned with fixed CSS so it stays in the corner on zoom/pan.
    # box-shadow and border-radius give the modern card appearance seen in
    # the sample. Each tier uses an inline circle swatch (not a character
    # entity) so the colour renders faithfully across all browsers.
    legend_html = """
    <div style="
      position: fixed;
      top: 16px; right: 16px;
      z-index: 9999;
      background: white;
      padding: 14px 18px;
      border-radius: 10px;
      box-shadow: 0 2px 14px rgba(0,0,0,0.18);
      font-family: Arial, sans-serif;
      font-size: 13px;
      line-height: 1.8;
      min-width: 195px;
    ">
      <div style="
        font-size: 14px; font-weight: bold;
        color: #333; margin-bottom: 8px;
        border-bottom: 1px solid #eee; padding-bottom: 6px;
      ">Violation Count Legend</div>

      <div style="display:flex; align-items:center; gap:9px; margin-bottom:3px;">
        <span style="
          display:inline-block; width:13px; height:13px;
          border-radius:50%; background:#2ca02c; flex-shrink:0;
        "></span>
        <span style="color:#444;">&le; 200 violations</span>
      </div>
      <div style="display:flex; align-items:center; gap:9px; margin-bottom:3px;">
        <span style="
          display:inline-block; width:13px; height:13px;
          border-radius:50%; background:#ff7f0e; flex-shrink:0;
        "></span>
        <span style="color:#444;">201 &ndash; 1000 violations</span>
      </div>
      <div style="display:flex; align-items:center; gap:9px;">
        <span style="
          display:inline-block; width:13px; height:13px;
          border-radius:50%; background:#d62728; flex-shrink:0;
        "></span>
        <span style="color:#444;">&gt; 1000 violations</span>
      </div>
    </div>
    """
    fmap.get_root().html.add_child(folium.Element(legend_html))
    return fmap

## §2. The dashboard renderer

In [11]:
def render_dashboard():
    """Fetch + render the full dashboard once. Designed to be called in a loop.

    All five charts plus a header strip are produced. The function is
    idempotent: it queries Mongo afresh on each call, so the loop just
    invokes it on a timer.

    Rendering strategy
    ------------------
    Plotly 6 dropped support for Jupyter Notebook < 7.  In Notebook 6.5.x
    environments, ``display(fig)`` silently produces no output.  The
    workaround is to serialise every Plotly figure to a self-contained HTML
    string via ``pio.to_html(..., include_plotlyjs='cdn')`` and render it
    with ``display(HTML(...))``.  The CDN tag is included in every chart div;
    modern browsers load the script once and cache it for subsequent charts.

    Folium maps must similarly be embedded via ``fmap._repr_html_()`` rather
    than passed directly to ``display()``, which falls back to the Python
    repr in Notebook 6.5.x.
    """
    since = (None if LOOKBACK_DAYS is None
             else dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
             - dt.timedelta(days=LOOKBACK_DAYS))

    viol_df = fetch_violations(violations_coll, since=since)
    cam_df  = fetch_camera_meta(cameras_coll)
    bucket  = choose_bucket(viol_df, default_bucket=BUCKET)
    agg     = aggregate_over_time(viol_df, bucket=bucket)
    points  = find_interesting_points(agg)

    # ----------------------------------------------------------------- Header
    total  = int(viol_df['violation_id'].count()) if not viol_df.empty else 0
    n_inst = int((viol_df['violation_type'] == 'INSTANTANEOUS').sum()) if not viol_df.empty else 0
    n_avg  = int((viol_df['violation_type'] == 'AVERAGE').sum()) if not viol_df.empty else 0
    n_cars = viol_df['car_plate'].nunique() if not viol_df.empty else 0
    refreshed = dt.datetime.now().strftime('%H:%M:%S')
    display(HTML(
        f'<div style="font-family:sans-serif;padding:8px 12px;background:#f6f8fa;'
        f'border:1px solid #d0d7de;border-radius:6px;">'
        f'<b>AWAS dashboard</b> | refreshed (local): {refreshed} | bucket: {bucket} | '
        f'total: <b>{total}</b> (instantaneous {n_inst} / average {n_avg}) | '
        f'unique cars: <b>{n_cars}</b></div>'
    ))

    if viol_df.empty:
        display(HTML('<i>No violations in MongoDB yet — start the producer notebooks.</i>'))
        return

    # ---------------------- Chart 1: Dual-axis time series with annotations
    fig1 = make_subplots(specs=[[{'secondary_y': True}]])
    fig1.add_trace(go.Bar(
        x=agg['bucket'], y=agg['count_total'], name='Violations / bucket',
        marker_color='rgba(31,119,180,0.55)',
    ), secondary_y=False)
    fig1.add_trace(go.Scatter(
        x=agg['bucket'], y=agg['avg_speed'], name='Avg speed (km/h)',
        mode='lines+markers', line=dict(color='crimson', width=2),
    ), secondary_y=True)
    fig1.add_trace(go.Scatter(
        x=agg['bucket'], y=agg['avg_speed_ma5'], name='Avg speed (5-bucket MA)',
        mode='lines', line=dict(color='purple', width=2, dash='dot'),
    ), secondary_y=True)
    if agg['p95'].notna().any():
        for c, dash, color in [
            ('p50', 'dot',     'rgba(46,160,67,0.7)'),
            ('p75', 'dashdot', 'rgba(255,166,0,0.7)'),
            ('p90', 'dash',    'rgba(255,128,0,0.7)'),
            ('p95', 'solid',   'rgba(207,34,46,0.7)'),
        ]:
            fig1.add_trace(go.Scatter(
                x=agg['bucket'], y=agg[c], name=c,
                line=dict(color=color, width=1, dash=dash),
                mode='lines', opacity=0.7,
            ), secondary_y=True)

    if not points.get('max_count', pd.DataFrame()).empty:
        r = points['max_count'].iloc[0]
        fig1.add_annotation(x=r['bucket'], y=r['count_total'], yref='y',
            text=f"MAX: {int(r['count_total'])}", showarrow=True, arrowhead=2,
            bgcolor='#cfe5ff', font=dict(size=11))
    if not points.get('min_count', pd.DataFrame()).empty:
        r = points['min_count'].iloc[0]
        fig1.add_annotation(x=r['bucket'], y=r['count_total'], yref='y',
            text=f"MIN: {int(r['count_total'])}", showarrow=True, arrowhead=2,
            bgcolor='#fff0c2', font=dict(size=11))
    if not points.get('max_speed', pd.DataFrame()).empty:
        r = points['max_speed'].iloc[0]
        fig1.add_annotation(x=r['bucket'], y=r['avg_speed'], yref='y2',
            text=f"MAX: {r['avg_speed']:0.1f} km/h", showarrow=True, arrowhead=2,
            bgcolor='#ffe2e2', font=dict(size=11))
    if not points.get('min_speed', pd.DataFrame()).empty:
        r = points['min_speed'].iloc[0]
        fig1.add_annotation(x=r['bucket'], y=r['avg_speed'], yref='y2',
            text=f"MIN: {r['avg_speed']:0.1f} km/h", showarrow=True, arrowhead=2,
            bgcolor='#d6f5d6', font=dict(size=11))

    for _, sp in points.get('spikes', pd.DataFrame()).iterrows():
        fig1.add_vrect(x0=sp['bucket'], x1=sp['bucket'] + pd.to_timedelta(bucket),
            fillcolor='red', opacity=0.10, line_width=0,
            annotation_text='SPIKE', annotation_position='top left',
            annotation=dict(font_size=9, font_color='red'))
    for _, dp in points.get('drops', pd.DataFrame()).iterrows():
        fig1.add_vrect(x0=dp['bucket'], x1=dp['bucket'] + pd.to_timedelta(bucket),
            fillcolor='blue', opacity=0.10, line_width=0,
            annotation_text='DROP', annotation_position='top left',
            annotation=dict(font_size=9, font_color='blue'))

    fig1.update_layout(
        title=f'Real-time Violations & Average Speed (bucket = {bucket})',
        legend=dict(orientation='h', y=1.12), height=440,
        template='plotly_white', margin=dict(l=50, r=50, t=70, b=40),
        barmode='overlay',
    )
    fig1.update_xaxes(title_text='Event time')
    fig1.update_yaxes(title_text='New violations per bucket', secondary_y=False)
    fig1.update_yaxes(title_text='Speed (km/h)', secondary_y=True)
    # Render via to_html — display(fig) produces no output with
    # Plotly 6 + Notebook 6.5.x (incompatible versions).
    display(HTML(pio.to_html(fig1, include_plotlyjs='cdn',
                             full_html=False, default_height=440)))

    # ------------------- Chart 2: Instantaneous vs Average stacked area
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(
        x=agg['bucket'], y=agg['count_inst'], name='Instantaneous',
        mode='lines', stackgroup='one', line=dict(color='#1f77b4'),
    ))
    fig2.add_trace(go.Scatter(
        x=agg['bucket'], y=agg['count_avg'], name='Average',
        mode='lines', stackgroup='one', line=dict(color='#d62728'),
    ))
    fig2.update_layout(
        title='Violation type contribution over time',
        xaxis_title='Event time', yaxis_title='Violations per bucket',
        template='plotly_white', height=320,
        margin=dict(l=50, r=50, t=60, b=40),
    )
    display(HTML(pio.to_html(fig2, include_plotlyjs='cdn',
                             full_html=False, default_height=320)))

    # ------------------- Chart 3: Per-camera violation counts (ending camera)
    by_cam = (viol_df.groupby('camera_id_end')['violation_id']
              .count().reset_index(name='count').sort_values('camera_id_end'))
    fig3 = go.Figure(go.Bar(
        x=by_cam['camera_id_end'].astype(str), y=by_cam['count'],
        text=by_cam['count'], textposition='outside',
        marker_color=['#2ca02c', '#ff7f0e', '#d62728'][:len(by_cam)],
    ))
    fig3.update_layout(
        title='Violations recorded per camera (ending camera)',
        xaxis_title='Camera id', yaxis_title='Violations',
        template='plotly_white', height=320,
        margin=dict(l=50, r=50, t=60, b=40),
    )
    display(HTML(pio.to_html(fig3, include_plotlyjs='cdn',
                             full_html=False, default_height=320)))

    # ------------------- Chart 4: Segment-level AVERAGE-speed violation counts
    avg_only = viol_df[viol_df['violation_type'] == 'AVERAGE']
    if not avg_only.empty:
        by_seg = (avg_only.groupby(['camera_id_start', 'camera_id_end'])['violation_id']
                  .count().reset_index(name='count'))
        by_seg['segment'] = by_seg.apply(
            lambda r: f"Cam {int(r.camera_id_start)} → Cam {int(r.camera_id_end)}",
            axis=1,
        )
        fig4 = go.Figure(go.Bar(
            x=by_seg['segment'], y=by_seg['count'],
            text=by_seg['count'], textposition='outside',
            marker_color=['#1f77b4', '#ff7f0e'][:len(by_seg)],
        ))
        fig4.update_layout(
            title='Average-speed violations by road segment',
            xaxis_title='Road segment', yaxis_title='Violations',
            template='plotly_white', height=320,
            margin=dict(l=50, r=50, t=60, b=40),
        )
        display(HTML(pio.to_html(fig4, include_plotlyjs='cdn',
                                 full_html=False, default_height=320)))
    else:
        display(HTML('<i>No AVERAGE-speed violations yet — segment chart will appear once stream-stream joins fire.</i>'))

    # ------------------- Chart 5: Folium map (cameras + segment polylines)
    # Explicit ._repr_html_() forces HTML MIME output; display(map_object)
    # falls back to the Python repr in Notebook 6.5.x.
    display(HTML(build_camera_map(cam_df, viol_df)._repr_html_()))


# One-shot render so the user sees something immediately, before the live loop.
render_dashboard()

## §3. Why these visualisations and these annotations?

| Chart | Operational question it answers | Why this annotation matters |
|---|---|---|
| **1. Dual-axis time series** | *When do violations peak, and is the peak driven by more cars or by faster cars?* | The bar shows volume; the line shows the speed regime. Spike shading (red) flags sudden onsets (incident clearing, weather break); drop shading (blue) flags sudden slowdowns. Dynamic p50 / p75 / p90 / p95 reference lines calibrate "abnormal" against the dataset's **own** distribution rather than a hard-coded threshold. The 5-bucket moving-average line smooths short-term jitter and is the standard visual cue for trend in time-series dashboards. |
| **2. Instantaneous vs Average stack** | *Are drivers braking only at the cameras (instantaneous low, average high), or speeding throughout (both high)?* | The core AWAS use case: if a checkpoint's instantaneous violations are low but its average violations are high, drivers are gaming the spot cameras and the point-to-point detection is doing exactly what it was built for. |
| **3. Per-camera bar** | *Which physical camera is the hottest?* | Drives enforcement scheduling, signage placement and re-calibration priority. |
| **4. Per-segment bar** | *Is one stretch of road generating more average-speed violations than another?* | Lets operators target road furniture (signage, rumble strips) at the right segment, not just the right camera. |
| **5. Folium map** | *Where on the road are violations geographically clustered?* | Non-technical stakeholders can see camera-level (markers) and segment-level (polylines) data on a real OSM basemap; colour-coded for at-a-glance triage. |

### Why dynamic percentiles instead of fixed thresholds?

If we hardcoded *"flag every bucket above 30 violations"*, the dashboard would either be cluttered on a busy day or empty on a quiet one. By computing p50/p75/p90/p95 *from the data currently in MongoDB*, the reference lines adapt automatically to the workload. The same idea underlies the spike/drop shading, which uses a rolling z-score rather than a static count.

## §4. Continuous live refresh

The cell below redraws the dashboard every `REFRESH_SECONDS` until either (a) you interrupt the kernel (■), or (b) `MAX_REFRESHES` ticks have elapsed (safety cap for unattended demos). At the default `REFRESH_SECONDS=5` and `MAX_REFRESHES=360`, the live loop runs for ~30 minutes.

In [12]:
try:
    for tick in range(MAX_REFRESHES):
        clear_output(wait=True)
        render_dashboard()
        time.sleep(REFRESH_SECONDS)
    print('\nMAX_REFRESHES reached - refresh halted. Re-run this cell to keep going.')
except KeyboardInterrupt:
    print('\nRefresh interrupted by user.')